# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | khaleelalaqel| 
| **Mehmet Kilinc 25135007** | ** |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [7]:
# Gedeelde imports (door iedereen te gebruiken/aan te vullen)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (6, 4)

RANDOM_STATE = 42 


**pd.set_option** zorgt ervoor dat pandas alle kolommen toont wanneer je een dataframe print, in plaats van kolommen af te korten met ...
<br>

**plt.rcParams** stelt de standaardgrootte in van elke grafiek die we met matplotlib maken

<br>

### Data inladen

In [8]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




------------

### 2.4 Evaluatiefuncties
Deze functies gebruiken we bij alle modellen:

- `get_scores()` geeft per patiënt een score: de kans op stroke (`predict_proba`) of, bij een SVM, de afstand tot het hypervlak (`decision_function`).
- `find_best_threshold()` zoekt de **beslisdrempel** met de hoogste F1-score. Een model voorspelt `stroke = 1` als de score boven die drempel ligt. Waarom dit belangrijk is, staat in paragraaf 5.1.
- `F1_BEST_SCORER` is de score die `GridSearchCV` gebruikt: de F1-score bij de beste drempel.
- `metrics_from_confusion()` rekent de metrics **zelf** uit met de formules uit paragraaf 4.2, zodat we kunnen laten zien hoe ze worden berekend.

In [ ]:
def get_scores(model, features):
    """Geeft per patiënt een score: kans op stroke of decision function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    return model.decision_function(features)


def f1_per_threshold(y_true, scores):
    """Berekent de F1-score voor iedere mogelijke beslisdrempel."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    # Het laatste punt van de curve hoort niet bij een drempel, want P=1 , R=0 en geen threshold. 
    f1_values = (2 * precision[:-1] * recall[:-1]
                 / np.maximum(precision[:-1] + recall[:-1], 1e-12))
    return thresholds, f1_values


def find_best_threshold(y_true, scores):
    """Geeft de drempel met de hoogste F1-score en die F1-score."""
    thresholds, f1_values = f1_per_threshold(y_true, scores)
    best = np.argmax(f1_values)
    return thresholds[best], f1_values[best]


def f1_at_best_threshold(y_true, scores):
    """Scorefunctie voor GridSearchCV: F1 bij de beste drempel."""
    return find_best_threshold(y_true, scores)[1]


F1_BEST_SCORER = make_scorer(
    f1_at_best_threshold,
    response_method=("predict_proba", "decision_function"))


def metrics_from_confusion(tn, fp, fn, tp):
    """Berekent de metrics zelf uit de vier cellen van de matrix."""
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    specificity = tn / (tn + fp) if tn + fp > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)
    return {"accuracy": accuracy, "precision": precision,
            "recall": recall, "specificity": specificity, "f1": f1}


def evaluate_predictions(y_true, y_pred, scores):
    """Combineert de metrics uit de confusion matrix met ROC- en PR-AUC."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    results = metrics_from_confusion(tn, fp, fn, tp)
    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"] = average_precision_score(y_true, scores)
    results.update({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
    return results